In [1]:
MAX_IMAGE_SIZE = 1120

In [2]:
import os
import re
import json
import fitz
import subprocess
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from tqdm.auto import tqdm
from glob import glob
from ast import literal_eval
from copy import deepcopy
from PIL import Image
from datasets import Dataset, load_dataset, load_from_disk

In [3]:
import io
def image_to_bytes(image):
    img_byte_arr = io.BytesIO()
    image.save(img_byte_arr, format='PNG')  # Specify the format (e.g., PNG or JPEG)
    img_byte_arr = img_byte_arr.getvalue()  # Get the byte data
    return img_byte_arr

In [4]:
tqdm.pandas()

In [5]:
def download_dataset(repo_url, local_dir):
    os.makedirs(clone_dir, exist_ok=True)
    
    # Command to clone the repository (Git LFS automatically handles LFS files)
    clone_command = ["git", "clone", repo_url, local_dir]
    
    # Run the git clone command
    try:
        subprocess.run(clone_command, check=True)
        print(f"Repository cloned successfully into {local_dir}")
    except subprocess.CalledProcessError as e:
        print(f"Error during cloning: {e}")

In [6]:
def get_pdf_images(path, image_size=MAX_IMAGE_SIZE):
    pdf_document = path
    pdf = fitz.open(pdf_document)
    
    images = []
    for page_num in range(len(pdf)):
        # Load the page
        page = pdf.load_page(page_num)
        
        # Render the page to a pixmap (convert to an image)
        pix = page.get_pixmap()
        
        # Convert to PIL image
        image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        if image.mode != 'RGB':
            image = image.convert('RGB')
        height, width = image.height, image.width
        if height>MAX_IMAGE_SIZE or width>MAX_IMAGE_SIZE:
            ratio = min(MAX_IMAGE_SIZE / height, MAX_IMAGE_SIZE / width)
            new_height = int(height * ratio)
            new_width = int(width * ratio)
            image = image.resize((new_width, new_height))
        images.append(image)
    if len(pdf)!= len(images):
        raise ValueError('len does not match')
    return images

In [11]:
def get_images(image_paths, image_size=MAX_IMAGE_SIZE):
    images = []
    page_numbers = []
    
    # Extract and store images along with their page numbers
    for image_path in image_paths:
        match = re.search(r'_p(\d+)\.jpg', image_path)
        if match:
            page_num = int(match.group(1))
            image = Image.open(image_path)
            
            # Convert to RGB if not already in RGB mode
            if image.mode != 'RGB':
                image = image.convert('RGB')

            height, width = image.height, image.width
            if height>MAX_IMAGE_SIZE or width>MAX_IMAGE_SIZE:
                ratio = min(MAX_IMAGE_SIZE / height, MAX_IMAGE_SIZE / width)
                new_height = int(height * ratio)
                new_width = int(width * ratio)
                image = image.resize((new_width, new_height))
            images.append((page_num, image))  # Store tuple of (page_num, image)
            page_numbers.append(page_num)
    
    # Sort images by page_num
    images.sort(key=lambda x: x[0])
    
    # Check for missing page numbers between 0 and max_page_num
    max_page_num = max(page_numbers)
    missing_pages = [i for i in range(max_page_num + 1) if i not in page_numbers]
    
    if missing_pages:
        raise ValueError(f"Missing page numbers: {missing_pages}")

    return [img for _, img in images]

In [8]:
repo_url = "https://huggingface.co/datasets/yubo2333/MMLongBench-Doc"
local_dir = "./data/MMLongBench-Doc/"
# download_dataset(repo_url, local_dir)

In [13]:
doc_ids_ignore = ['2312.09390v1.pdf', 'PS_2018.01.09_STEM_FINAL.pdf', 'f1f5242528411b262be447e61e2eb10f.pdf', 'san-francisco-11-contents.pdf', 'mi_phone.pdf', '2303.08559v2.pdf', '379f44022bb27aa53efd5d322c7b57bf.pdf', 'edb88a99670417f64a6b719646aed326.pdf', 'f86d073b0d735ac873a65d906ba82758.pdf']

file_size = 100
file_num = 0

# Load dataset
dataset = load_dataset(local_dir, split='train')
dataset = dataset.filter(lambda example: example['doc_id'] not in doc_ids_ignore)

# Initialize DataFrame with columns
dataset_df = pd.DataFrame(columns=['image_paths', 'doc_types', 'questions', 'labels', 'label_types', 'answers'], index=list(set(dataset['doc_id'])))

for example in tqdm(dataset):
    labels = literal_eval(example['evidence_pages'])
    labels = list(set([int(label)-1 for label in labels]))
    
    if not labels or -1 in labels:
        continue

    # Initialize rows for each document id
    if type(dataset_df.loc[example['doc_id'], 'image_paths']) != list:
        pdf_path = os.path.join(local_dir, 'documents', example['doc_id'])
        if not os.path.exists(pdf_path):
            continue
        
        dataset_df.loc[example['doc_id'], 'image_paths'] = pdf_path
        dataset_df.loc[example['doc_id'], 'doc_types'] = []
        dataset_df.loc[example['doc_id'], 'questions'] = []
        dataset_df.loc[example['doc_id'], 'labels'] = []
        dataset_df.loc[example['doc_id'], 'label_types'] = []
        dataset_df.loc[example['doc_id'], 'answers'] = []

    # Populate lists for each doc_id
    dataset_df.loc[example['doc_id'], 'doc_types'] += [example['doc_type']]
    dataset_df.loc[example['doc_id'], 'questions'] += [example['question']]
    dataset_df.loc[example['doc_id'], 'labels'] += [labels]
    dataset_df.loc[example['doc_id'], 'label_types'] += [literal_eval(example['evidence_sources'])]
    dataset_df.loc[example['doc_id'], 'answers'] += [[example['answer']]]

# Reset the index and specify the dataset name
dataset_df = dataset_df.reset_index(names=['doc_id'])
dataset_df['dataset'] = 'mm_long_bench'

# Batch processing and saving to parquet files
for i in range(0, len(dataset_df), file_size):
    temp_dataset_df = dataset_df.iloc[i:i+file_size].copy()
    temp_dataset_df['images'] = temp_dataset_df['image_paths'].progress_apply(lambda paths: get_pdf_images(paths, image_size=MAX_IMAGE_SIZE))
    temp_dataset_df['image_bytes'] = temp_dataset_df['images'].progress_apply(lambda images: [image_to_bytes(image) for image in images])
    
    # Drop unnecessary columns
    temp_dataset_df = temp_dataset_df.drop(['image_paths', 'images', 'doc_id'], axis=1)
    
    # Write to Parquet
    parquet_file_name = f'./eval_data/mm_long_bench/train-{file_num:05d}.parquet'
    os.makedirs(os.path.dirname(parquet_file_name), exist_ok=True)
    table = pa.Table.from_pandas(temp_dataset_df)
    pq.write_table(table, parquet_file_name)
    file_num += 1


  0%|          | 0/1022 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/26 [00:00<?, ?it/s]

  0%|          | 0/26 [00:00<?, ?it/s]

In [1]:
#mpdocvqa

In [ ]:
file_size = 100
file_num = 0

with open("./data/MP-DOCVQA/val.json", "r") as f:
    mp_docvqa_val = json.load(f)

dataset_df = pd.DataFrame(columns=['dataset', 'image_paths', 'doc_types', 'questions', 'labels', 'label_types','answers'], index=list(set([data['doc_id'] for data in mp_docvqa_val['data']])))
for example in tqdm(mp_docvqa_val['data']):
    labels = [example['answer_page_idx']]
    if type(dataset_df.loc[example['doc_id'], 'image_paths']) != list:
        image_paths = glob(f"./data/MP-DOCVQA/images/{example['doc_id']}*")
        dataset_df.loc[example['doc_id'], 'image_paths'] = image_paths
        dataset_df.loc[example['doc_id'], 'doc_types'] = []
        dataset_df.loc[example['doc_id'], 'questions'] = []
        dataset_df.loc[example['doc_id'], 'labels'] = []
        dataset_df.loc[example['doc_id'], 'label_types'] = []
        dataset_df.loc[example['doc_id'], 'answers'] = []
        
    dataset_df.loc[example['doc_id'], 'questions'] = dataset_df.loc[example['doc_id'], 'questions'] + [example['question']]
    dataset_df.loc[example['doc_id'], 'labels'] = dataset_df.loc[example['doc_id'], 'labels'] + [labels]
    dataset_df.loc[example['doc_id'], 'answers'] = dataset_df.loc[example['doc_id'], 'answers'] + [example['answers']]
    dataset_df.loc[example['doc_id'], 'label_types'] = dataset_df.loc[example['doc_id'], 'label_types'] + [['None']]
    dataset_df.loc[example['doc_id'], 'doc_types'] = dataset_df.loc[example['doc_id'], 'doc_types'] + ['None']

# Reset the index and specify the dataset name
dataset_df = dataset_df.reset_index(names=['doc_id'])
dataset_df['dataset'] = 'mp_docvqa'

# Batch processing and saving to parquet files
for i in range(0, len(dataset_df), file_size):
    temp_dataset_df = dataset_df.iloc[i:i+file_size].copy()
    temp_dataset_df['images'] = temp_dataset_df['image_paths'].progress_apply(lambda paths: get_images(paths, image_size=MAX_IMAGE_SIZE))
    temp_dataset_df['image_bytes'] = temp_dataset_df['images'].progress_apply(lambda images: [image_to_bytes(image) for image in images])
    
    # Drop unnecessary columns
    temp_dataset_df = temp_dataset_df.drop(['image_paths', 'images', 'doc_id'], axis=1)
    
    # Write to Parquet
    parquet_file_name = f'./eval_data/mp_docvqa/train-{file_num:05d}.parquet'
    os.makedirs(os.path.dirname(parquet_file_name), exist_ok=True)
    table = pa.Table.from_pandas(temp_dataset_df)
    pq.write_table(table, parquet_file_name)
    file_num += 1


In [16]:
dataset_df['dataset'] = 'mp_docvqa'

# Batch processing and saving to parquet files
for i in range(0, len(dataset_df), file_size):
    temp_dataset_df = dataset_df.iloc[i:i+file_size].copy()
    temp_dataset_df['images'] = temp_dataset_df['image_paths'].progress_apply(lambda paths: get_images(paths, image_size=MAX_IMAGE_SIZE))
    temp_dataset_df['image_bytes'] = temp_dataset_df['images'].progress_apply(lambda images: [image_to_bytes(image) for image in images])
    
    # Drop unnecessary columns
    temp_dataset_df = temp_dataset_df.drop(['image_paths', 'images', 'doc_id'], axis=1)
    
    # Write to Parquet
    parquet_file_name = f'./eval_data/mp_docvqa/train-{file_num:05d}.parquet'
    os.makedirs(os.path.dirname(parquet_file_name), exist_ok=True)
    table = pa.Table.from_pandas(temp_dataset_df)
    pq.write_table(table, parquet_file_name)
    file_num += 1

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

...

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

In [3]:
eval_dataset = load_dataset('./eval_data/mp_docvqa/', split='train')

In [5]:
eval_dataset = eval_dataset.sort('doc_id')

In [10]:
len(eval_dataset[:1]['image_bytes'][0])

7

In [18]:
import os
from glob import glob
from datasets import load_dataset, concatenate_datasets

In [19]:
eval_datasets = []
for dataset_path in glob('./eval_data/*'):
    dataset_name = os.path.basename(dataset_path)
    eval_dataset = load_dataset(dataset_path, split='train')
    eval_dataset.push_to_hub('jwengr/document-vqa-evaluation',config_name=dataset_name, token='')

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Generating train split: 0 examples [00:00, ? examples/s]

Uploading the dataset shards:   0%|          | 0/22 [00:00<?, ?it/s]

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/618 [00:00<?, ?B/s]

C:\Users\dust\anaconda3\envs\py311_torch2\Lib\site-packages\huggingface_hub\file_download.py:147: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dust\.cache\huggingface\hub\datasets--jwengr--document-vqa-evaluation. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
